# How to build a linear factor model

Algorithmic trading strategies use linear factor models to quantify the relationship between the return of an asset and the sources of risk that represent the main drivers of these returns. Each factor risk carries a premium, and the total asset return can be expected to correspond to a weighted average of these risk premia.

There are several practical applications of factor models across the portfolio management process from construction and asset selection to risk management and performance evaluation. The importance of factor models continues to grow as common risk factors are now tradeable:

- A summary of the returns of many assets by a much smaller number of factors reduces the amount of data required to estimate the covariance matrix when optimizing a portfolio
- An estimate of the exposure of an asset or a portfolio to these factors allows for the management of the resultant risk, for instance by entering suitable hedges when risk factors are themselves traded
- A factor model also permits the assessment of the incremental signal content of new alpha factors
- A factor model can also help assess whether a manager's performance relative to a benchmark is indeed due to skill in selecting assets and timing the market, or if instead, the performance can be explained by portfolio tilts towards known return drivers that can today be replicated as low-cost, passively managed funds without incurring active management fees

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import os

from statsmodels.api import OLS, add_constant
import pandas_datareader.data as web

from linearmodels.asset_pricing import LinearFactorModel

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
sns.set_style('whitegrid')

## Get data

Fama and french make update risk factor and research portfolio data availabel through their website and you can use the pandas datareader package to obtin the data.

### Risk Factors

In particular, we will be using the five Fama—French factors that result from sorting stocks first into three size groups and then into two for each of the remaining three firm-specific factors.

Hence, the factors involve three sets of value-weighted portfolios formed as 3 x 2 sorts on size and book-to-market, size and operating profitability, and size and investment. The risk factor values computed as the average returns of the portfolios (PF) as outlined in the following table:

| Label | Name                     | Description |
|------:|--------------------------|-------------|
| SMB   | Small Minus Big           | Average return on the nine small stock portfolios minus the average return on the nine big stock portfolios |
| HML   | High Minus Low            | Average return on the two value portfolios minus the average return on the two growth portfolios |
| RMW   | Robust Minus Weak         | Average return on the two robust operating profitability portfolios minus the average return on the two weak operating profitability portfolios |
| CMA   | Conservative Minus Aggressive | Average return on the two conservative investment portfolios minus the average return on the two aggressive investment portfolios |
| Rm − Rf | Excess return on the market | Value-weight return of all firms incorporated in the US and listed on the NYSE, AMEX, or NASDAQ at the beginning of month *t* with “good” data for *t* minus the one-month Treasury bill rate |


We will use returns at a daily frequency that we obtain for the period 2010 – 2017 as follows:

In [7]:
ff_factor = 'F-F_Research_Data_5_Factors_2x3_daily'
ff_factor_data = web.DataReader(ff_factor, 'famafrench', start='2010', end='2017-12')[0]
ff_factor_data.head()

C:\Users\Marti\AppData\Local\Temp\ipykernel_25572\4169315089.py:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factor_data = web.DataReader(ff_factor, 'famafrench', start='2010', end='2017-12')[0]


,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
2010-01-04,1.69,0.79,1.14,-0.17,0.21,0.0
2010-01-05,0.31,-0.42,1.22,-0.18,0.18,0.0
2010-01-06,0.13,-0.14,0.55,-0.05,0.20,0.0
2010-01-07,0.40,0.25,0.96,-0.66,0.22,0.0
2010-01-08,0.33,0.31,0.02,0.23,-0.40,0.0


In [8]:
ff_factor_data.describe()

,Mkt-RF,SMB,HML,RMW,CMA,RF
count,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.0
mean,0.056580,0.003806,-0.001946,0.004880,0.001479,0.0
std,0.960457,0.519405,0.481534,0.345935,0.288261,0.0
min,-6.960000,-1.990000,-1.870000,-1.800000,-1.290000,0.0
25%,-0.350000,-0.320000,-0.297500,-0.200000,-0.180000,0.0
50%,0.070000,0.010000,-0.020000,0.000000,0.000000,0.0
75%,0.530000,0.317500,0.270000,0.197500,0.160000,0.0
max,4.970000,3.610000,2.440000,1.790000,1.970000,0.0


## Portfolios

Fama and French also make available numerous portfolios that we can illustrate the estimation of the factor exposures, as well as the value of the risk premia available in the market for a given time period. We will use a panel of the 17 industry portfolios at a daily frequency.

We will subtract the risk-free rate from the returns because the factor model works with excess returns: